In [ ]:
#Install dependencies
%pip install anthropic python-dotenv

In [ ]:
# load env variables   
from dotenv import load_dotenv

load_dotenv()  # take environment variables from .env.

In [27]:
from anthropic import Anthropic
import os

client = Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
for m in client.models.list():
    print(m.id)

claude-haiku-5-5
claude-sonnet-5-5
claude-opus-5-5
claude-fable-5-1
claude-opus-5
claude-sonnet-5
claude-fable-5
claude-opus-4-8
claude-opus-4-7
claude-sonnet-4-6
claude-opus-4-6
claude-opus-4-5-20251101
claude-haiku-4-5-20251001


In [ ]:
# create an API client
from anthropic import Anthropic

client = Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
model = "claude-sonnet-5-5"

In [ ]:
#from email.mime import message


def add_user_message(messages,text):
    messages.append({"role": "user", "content": text})
    return messages

def add_assistant_message(messages,text):
    messages.append({"role": "assistant", "content": text})
    return messages



def chat(messages):
    messages = client.messages.create(
    model=model,
    max_tokens=1024,
    messages=messages,
)
    return "".join(block.text for block in messages.content if block.type == "text")

In [ ]:
#make a list of messages to send to the model
messages = []

#add in the initial user message
messages = add_user_message(messages, "What is the capital of France?")

messages

#pass the messages to the chat function and get the response
response = chat(messages)

response

#take the answer and add it to the messages list as an assistant message
messages = add_assistant_message(messages, response)

messages

#add user follow up message
messages = add_user_message(messages, "Write another sentence")

#call the chat function again with the updated messages list
response = chat(messages)
response

In [ ]:
message = client.messages.create(
    model=model,
    max_tokens=1024,
    messages=[
        {"role": "user", "content": "What is the history of France? Answer in 1 sentence."}
    ],
)

message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {"role": "user", "content": "Write another sentence."}
    ],
)

print("".join(block.text for block in message.content if block.type == "text"))

#message.content[0].text

In [ ]:
#exercise to create a simple chat loop that takes user input and sends it to the model, then prints the response

def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})
    return messages

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})
    return messages

def chat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=1024,
        messages=messages,
    )
    return "".join(block.text for block in message.content if block.type == "text")


messages = []

while True:
    user_input = input("You: ").strip()

    if user_input.lower() == "exit":
        break
    if not user_input:          # ignore blank input
        continue

    messages = add_user_message(messages, user_input)
    response = chat(messages)
    print("You:", user_input)
    print("Assistant:", response)
    messages = add_assistant_message(messages, response)

In [28]:
#using a system prompt to set the behavior of the assistant

def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})
    return messages

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})
    return messages

def chat(messages, system=None, temperature=1.0):
    params={
        "model": "claude-sonnet-4-6",
        "max_tokens": 1024,
        "messages": messages,
        "system": system,
        "temperature": temperature
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return "".join(block.text for block in message.content if block.type == "text")

   

In [23]:
messages = []

add_user_message(messages, "Write a python function to calculate the factorial of a number.")

answer = chat(messages, system="You are a helpful assistant that writes concise python code." )

answer

'```python\ndef factorial(n: int) -> int:\n    if not isinstance(n, int) or n < 0:\n        raise ValueError("n must be a non-negative integer")\n    result = 1\n    for i in range(2, n + 1):\n        result *= i\n    return result\n```\n\n**Example:**\n```python\nprint(factorial(5))  # 120\nprint(factorial(0))  # 1\n```\n\n**Notes:**\n- Uses iteration, so there\'s no recursion depth limit.\n- Raises `ValueError` for negative or non-integer input.\n- Alternative: `math.factorial(n)` from the standard library does the same thing and is implemented in C.'

In [34]:
messages = []

add_user_message(messages, "Write a short fictional series idea.")

answer = chat(messages, system="You are a helpful assistant",temperature=0.0)

answer

"# **The Hollow Hours**\n*A Psychological Mystery Series*\n\n---\n\n## The Premise\n\nEvery night between 3 and 4 AM, a small coastal town goes completely silent. No wind. No waves. No animals. And every morning, one resident wakes up somewhere they don't remember going.\n\n---\n\n## The Setup\n\n**Mara Voss**, a sleep researcher who hasn't dreamed in seven years, moves to Crestfall, Oregon after a professional scandal. She expects quiet. Instead, she finds a town full of people who have quietly accepted something deeply wrong — and a locked lighthouse no one has entered in thirty years.\n\n---\n\n## The Hook\n\nEach episode follows a different resident during *their* lost hour, piecing together a larger picture of what Crestfall is hiding beneath its charming surface.\n\n---\n\n## The Tone\n\nThink *Twin Peaks* meets *The Leftovers* — eerie and grounded, more about grief and secrets than monsters.\n\n---\n\n## Season Arc\n\nThe lost hours aren't random. They're **counting down** to so

In [37]:
#how to use straming to get the response from the model as it is generated

messages = []

add_user_message(messages, "Write a python function to calculate the factorial of a number.")

stream = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_011Cg8SpTQ6WVoVUsqWfPSDv', container=None, content=[], model='claude-sonnet-4-6', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=19, output_tokens=1, output_tokens_details=None, server_tool_use=None, service_tier='standard', speed='standard'), diagnostics=None), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='##', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' Factorial Function', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' in', type='text_del

In [41]:
#alternative way for streaming the response from the model as it is generated

messages = []

add_user_message(messages, "Write a python function to calculate the factorial of a number.")

with client.messages.stream(
    model="claude-sonnet-4-6",
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        #print(text, end="", flush=True)
        pass

    stream.get_final_message()  # Wait for the final response to be received

    message = stream.get_final_message()